### Example Fixed Project

The rest of this tutorial uses pre compiled ORBIT configs that are stored as .yaml files in the '~/configs/ folder. There are load and save methods available in ORBIT for working with .yaml files. These example projects each exhibit different functionalities within ORBIT. Using these examples and combinations of them, most project configurations can be modeled. 

In [1]:
import os
import pandas as pd
from ORBIT import ProjectManager, load_config

weather = pd.read_csv("data/example_weather.csv", parse_dates=["datetime"])\
            .set_index("datetime")

Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.

### Load the project configuration

In [2]:
fixed_config = load_config("configs/example_floating_project_600MW.yaml")  # Configs can be loaded with absolute or relative paths

print(type(fixed_config))                                         # They are loaded in as dictionaries.

print(f"Num turbines: {fixed_config['plant']['num_turbines']}")   # Once a configuration is loaded, different parameters can  
print(f"Turbine: {fixed_config['turbine']}")                      # be accessed using dict access.
print(f"\nSite: {fixed_config['site']}")

<class 'dict'>
Num turbines: 50
Turbine: 12MW_generic

Site: {'depth': 739, 'distance': 189, 'distance_to_landfall': 36, 'mean_windspeed': 8.41}


### Phases

This fixed project represents a generic Offshore Wind farm with 50 6MW turbines. It includes 5 design modules and 6 installation modules as seen below. This is a common set of modules to run for a fixed bottom project. This config will model the procurement and installation of monopiles, scour protection, array system, export system, offshore substation and the turbines.

In [3]:
print(f"Design phases: {fixed_config['design_phases']}")
print(f"\nInstall phases: {list(fixed_config['install_phases'].keys())}")

Design phases: ['ArraySystemDesign', 'ElectricalDesign', 'MooringSystemDesign', 'SemiSubmersibleDesign']

Install phases: ['ArrayCableInstallation', 'ExportCableInstallation', 'MooredSubInstallation', 'MooringSystemInstallation', 'OffshoreSubstationInstallation']


### Run

This project is always being modeled with the example weather project supplied that is representative of US East Coast wind farm locations.

In [4]:
project = ProjectManager(fixed_config, weather=weather)
project.run()

ORBIT library intialized at 'C:\ORBIT_before_cost_updates\ORBIT\library'


`trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.DeprecationWarning: C:\ORBIT_before_cost_updates\ORBIT\ORBIT\manager.py:730
landfall dictionary will be deprecated and moved into [export_system][landfall].DeprecationWarning: C:\ORBIT_before_cost_updates\ORBIT\ORBIT\phases\install\quayside_assembly_tow\moored.py:94
support_vessel will be deprecated and replaced with towing_vessels and ahts_vessel in the towing groups.
['towing_vessl_groups]['station_keeping_vessels'] will be deprecated and replaced with ['towing_vessl_groups]['ahts_vessels'].


### Top Level Outputs

ProjectManager offers several high level result categories:
- Installation CapEx
- System CapEx (procurement of BOS subcomponents)
- Turbine CapEx
- Soft CapEx (project management costs)
- Total CapEx
- Total installation time
- etc.

In [5]:
print(f"Installation CapEx:  {project.installation_capex/1e6:.0f} M")
print(f"System CapEx:        {project.system_capex/1e6:.0f} M")
print(f"Turbine CapEx:       {project.turbine_capex/1e6:.0f} M")
print(f"Soft CapEx:          {project.soft_capex/1e6:.0f} M")
print(f"Total CapEx:        {project.total_capex/1e6:.0f} M")

print(f"\nInstallation Time: {project.installation_time:.0f} h")

Installation CapEx:  265 M
System CapEx:        1227 M
Turbine CapEx:       1073 M
Soft CapEx:          543 M
Total CapEx:        3259 M

Installation Time: 20552 h


### CapEx Breakdown

In [6]:
# The breakdown of project costs by module is available  at 'capex_breakdown'
data = project.capex_detailed_soft_capex_breakdown_per_kw

# Convert the dictionary into a pandas DataFrame
df = pd.DataFrame(list(data.items()), columns=['Category', 'Value'])

# Add a "Total" row
total_row = pd.DataFrame([['Total', df['Value'].sum()]], columns=['Category', 'Value'])
df = pd.concat([df, total_row], ignore_index=True)

# Display the DataFrame
df.to_csv("floating_costs.csv")
print(df)

                            Category        Value
0                       Array System    99.710098
1                      Export System   211.035774
2                       Substructure  1051.182728
3                     Mooring System   459.354567
4                Offshore Substation   223.384178
5          Array System Installation   131.997462
6         Export System Installation    62.995612
7          Substructure Installation   127.438989
8        Mooring System Installation   109.880032
9   Offshore Substation Installation     8.886012
10                           Turbine  1789.000000
11                           Project   252.083333
12            Construction Insurance    52.059911
13                   Decommissioning    77.209669
14                     Commissioning    52.059911
15           Procurement Contingency   234.930664
16          Installation Contingency   152.213347
17            Construction Financing   336.419142
18                             Total  5431.841428


### Installation Actions

In [7]:
df = pd.DataFrame(project.actions)    # The project simulation logs are also available for all modules
df

,cost_multiplier,agent,action,duration,cost,level,time,phase,location,phase_name,max_waveheight,max_windspeed,transit_speed,site_depth,num_vessels,num_ahts_vessels
0,0.5,Array Cable Installation Vessel,Mobilize,72.000,3.375000e+05,ACTION,0.000,ArrayCableInstallation,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0.5,Export Cable Installation Vessel,Mobilize,72.000,3.375000e+05,ACTION,0.000,ExportCableInstallation,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,Onshore Construction,Onshore Construction,0.000,4.001844e+06,ACTION,0.000,ExportCableInstallation,Landfall,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1.0,Mooring System Installation Vessel,Mobilize,168.000,7.000000e+05,ACTION,0.000,MooringSystemInstallation,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0.5,Heavy Lift Vessel,Mobilize,72.000,7.500000e+05,ACTION,0.000,OffshoreSubstationInstallation,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2741,NaN,Mooring System Installation Vessel,Install Mooring Line,3.695,1.539583e+04,ACTION,6737.710,MooringSystemInstallation,NaN,MooringSystemInstallation,NaN,NaN,NaN,NaN,NaN,NaN
2742,NaN,Mooring System Installation Vessel,Position Onsite,2.000,8.333333e+03,ACTION,6739.710,MooringSystemInstallation,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2743,NaN,Mooring System Installation Vessel,Perform Mooring Site Survey,4.000,1.666667e+04,ACTION,6743.710,MooringSystemInstallation,NaN,MooringSystemInstallation,NaN,NaN,NaN,NaN,NaN,NaN
2744,NaN,Mooring System Installation Vessel,Install Suction Pile Anchor,14.695,6.122917e+04,ACTION,6758.405,MooringSystemInstallation,NaN,MooringSystemInstallation,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
# These logs can be sorted by phase by using DataFrame operations

turbine_install = df.loc[df['phase']=="TurbineInstallation"]
turbine_install

,cost_multiplier,agent,action,duration,cost,level,time,phase,location,phase_name,max_waveheight,max_windspeed,transit_speed,site_depth,num_vessels,num_ahts_vessels


In [9]:
# Operations can also be grouped to see a total amount of time spend on each operation

turbine_install.groupby(["action"]).sum()['duration']

Series([], Name: duration, dtype: float64)